# CORREÇÃO - CLÁUDIO NEVES — Samuel, eu refiz o seu gabarito, veja que falta para nota 10,00

Este notebook mostra **exatamente o que falta** no seu projeto para ir de **7,00 → 10,00**.

São **4 ajustes** de 1 a 3 linhas cada. Todo o restante do projeto está correto.

| Critério | Nota atual | Com ajuste | Diferença |
|---|---|---|---|
| Manipulação de CSV | N2 — 0,75 | N3 — 1,50 | **+0,75** |
| Nulos e Condicionais | N2 — 0,75 | N3 — 1,50 | **+0,75** |
| Regras de Negócio e Datas | N2 — 0,75 | N3 — 1,50 | **+0,75** |
| Geração de Estatísticas | N2 — 0,75 | N3 — 1,50 | **+0,75** |
| **Total** | **7,00** | **10,00** | **+3,00** |

In [ ]:
import sys
sys.path.append(r"C:\Users\s2bcl\AppData\Roaming\Python\Python313\site-packages")

import kagglehub
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

print("Bibliotecas carregadas com sucesso!")

## AJUSTE 1 — Manipulação de Arquivos CSV

**Problema:** `read_csv` sem `encoding` e `decimal`, e sem gerar CSV de saída ao final.  
**Solução:** Adicionar os parâmetros e exportar a base limpa com `to_csv`.

In [ ]:
def extrair_dados():
    path = kagglehub.dataset_download("namespaiva/base-varejo")
    csv_path = Path(path) / "Base Varejo.csv"

    # ANTES — nota N2 (0,75):
    # df = pd.read_csv(csv_path, sep=';')

    # DEPOIS — nota N3 (1,50): adicionar encoding e decimal
    df = pd.read_csv(csv_path, sep=';', encoding='cp1252', decimal=',')
    return df

df = extrair_dados()

print("Dataset carregado com sucesso!\n")
print(f"O dataframe tem {df.shape[0]} linhas e {df.shape[1]} colunas.\n")
print(df.head().to_string())
print("\nColunas do dataset:", df.columns.tolist())
print("\nTipos de dados das colunas:\n", df.dtypes)

In [ ]:
# ANÁLISE DA BASE DE DADOS (igual ao original do Samuel)

print("\nEstatísticas descritivas do dataset:\n")
print(df.describe(include="all").round(2).to_string())

print("\nQuantidade de valores nulos por coluna:\n")
print(df.isnull().sum())

print("\nValores únicos por coluna:\n")
for coluna in df.columns:
    print(f"{coluna}: {df[coluna].nunique()} valores únicos")

linhas_duplicadas = df.duplicated().sum()
print(f"\nQuantidade de linhas duplicadas: {linhas_duplicadas}")

## AJUSTE 2 — Tratamento de Nulos e Condicionais

**Problema:** Estratégia uniforme para todas as colunas — só `replace` e `drop`.  
**Solução:** Estratégias **diferentes por tipo de coluna**: `fillna(mediana)` para numéricas, `replace` para categóricas.

In [ ]:
# LIMPEZA E TRATAMENTO DOS DADOS

colunas_irrelevantes = ['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
df_limpo = df.copy()
df_limpo = df_limpo.drop(columns=colunas_irrelevantes)
print("Colunas Unnamed removidas.")

# Converter DATA para datetime (igual ao original)
df_limpo['DATA'] = pd.to_datetime(df_limpo['DATA'], errors='coerce', format='%d/%m/%Y')
print(f"Datas inválidas após conversão: {df_limpo['DATA'].isnull().sum()}")

# Remover duplicatas (igual ao original)
df_limpo = df_limpo.drop_duplicates()
print(f"Formato após remoção de duplicatas: {df_limpo.shape}")

# Tratar PR_CAT — coluna CATEGÓRICA → substituir texto inválido
df_limpo["PR_CAT"] = df_limpo["PR_CAT"].replace("#N/D", "SEM CATEGORIA")

# ANTES — nota N2 (0,75): só o replace acima, sem tratar CL_FHL

# DEPOIS — nota N3 (1,50): tratar CL_FHL com mediana (coluna NUMÉRICA → fillna)
mediana_filhos = df_limpo['CL_FHL'].median()
df_limpo['CL_FHL'] = df_limpo['CL_FHL'].fillna(mediana_filhos)
print(f"Nulos em CL_FHL tratados com mediana: {mediana_filhos}")

print(df_limpo["PR_CAT"].value_counts())

In [ ]:
# TRATAMENTO E EXPLORAÇÃO DE DADOS CATEGÓRICOS (igual ao original)

estado_civil_map = {
    1: "Casado/União Estável",
    2: "Divorciado",
    3: "Separado",
    4: "Solteiro",
    5: "Viúvo"
}
df_limpo['CL_EC_CATEGORIA'] = df_limpo['CL_EC'].map(estado_civil_map)

colunas_categoricas = ['CL_EC_CATEGORIA', 'CL_GENERO', 'CL_SEG', 'PR_CAT', 'PR_NOME']
for coluna in colunas_categoricas:
    df_limpo[coluna] = df_limpo[coluna].astype('category')

print("\nTipos de dados após conversão:", df_limpo.dtypes)

for coluna in colunas_categoricas:
    print(f"\nValores únicos na coluna '{coluna}':\n")
    print(df_limpo[coluna].value_counts().head(10).to_string())

In [ ]:
# AJUSTE 1 (continuação) — EXPORTAR CSV DE SAÍDA

# ANTES — nota N2: sem nenhum to_csv()

# DEPOIS — nota N3: exportar a base limpa
df_limpo.to_csv('base_varejo_limpa.csv', index=False, sep=';', encoding='cp1252')
print("Base limpa exportada: base_varejo_limpa.csv")
print(f"Linhas exportadas: {len(df_limpo)}")

In [ ]:
# ESTATÍSTICAS DESCRITIVAS PARA A COLUNA CL_FHL (igual ao original)

print("\nEstatísticas descritivas para a coluna 'CL_FHL':\n")
print(df_limpo['CL_FHL'].describe().round(2).to_string())

moda_cl_fhl = df_limpo['CL_FHL'].mode()
print(f"\nModa da coluna 'CL_FHL': {moda_cl_fhl.values[0]}")

## AJUSTE 3 — Regras de Negócio e Datas

**Problema:** A coluna `DATA` foi convertida para `datetime`, mas nunca foi usada em nenhuma análise.  
**Solução:** Criar colunas derivadas (`MES`, `ANO`) e usá-las em um `groupby` temporal.

In [ ]:
# ANTES — nota N2 (0,75): DATA convertida mas nunca utilizada para análise

# DEPOIS — nota N3 (1,50): criar colunas derivadas e usar em agrupamento temporal
df_limpo['MES'] = df_limpo['DATA'].dt.month
df_limpo['ANO'] = df_limpo['DATA'].dt.year

vendas_por_mes = (
    df_limpo.groupby('MES')['PR_ID']
    .count()
    .reset_index()
    .rename(columns={'PR_ID': 'Quantidade de Vendas', 'MES': 'Mês'})
    .sort_values('Mês')
)

print("\nVendas por mês:\n")
print(vendas_por_mes.to_string(index=False))

In [ ]:
# PADRÕES DE AGRUPAMENTO (igual ao original — já estava N3)

vendas_por_genero = (
    df_limpo.groupby('CL_GENERO')['PR_ID'].count().reset_index()
    .rename(columns={'PR_ID': 'Quantidade Total de Vendas', 'CL_GENERO': 'Gênero'})
    .sort_values(by='Quantidade Total de Vendas', ascending=False)
)
print("\nQuantidade total de vendas por gênero:\n")
print(vendas_por_genero.reset_index(drop=True).to_string())

vendas_por_categoria = (
    df_limpo.groupby('PR_CAT')['PR_ID'].count().reset_index()
    .rename(columns={'PR_ID': 'Quantidade Total de Vendas', 'PR_CAT': 'Categoria de Produto'})
    .sort_values(by='Quantidade Total de Vendas', ascending=False)
)
print("\nQuantidade total de vendas por categoria de produto:\n")
print(vendas_por_categoria.reset_index(drop=True).to_string())

produtos_mais_comprados_por_genero = (
    df_limpo.groupby(['CL_GENERO', 'PR_NOME'])['PR_ID'].count().reset_index()
    .rename(columns={'PR_ID': 'Quantidade Total de Vendas', 'CL_GENERO': 'Gênero', 'PR_NOME': 'Produto'})
    .sort_values(by='Quantidade Total de Vendas', ascending=False)
)
print("\nProdutos mais comprados por gênero (top 10):\n")
print(produtos_mais_comprados_por_genero.head(10).reset_index(drop=True).to_string())

pivot_vendas_genero_categoria = (
    df_limpo.pivot_table(index='CL_GENERO', columns='PR_CAT',
                         values='PR_ID', aggfunc='count', fill_value=0)
    .reset_index().rename(columns={'CL_GENERO': 'Gênero'})
)
print("\nQuantidade de vendas por gênero e categoria de produto:\n")
print(pivot_vendas_genero_categoria.reset_index(drop=True).to_string())

## AJUSTE 4 — Geração de Estatísticas Básicas

**Problema:** Só `describe()` e moda — sem nenhuma visualização gráfica.  
**Solução:** Adicionar pelo menos 1 gráfico com `matplotlib`.

In [ ]:
# ANTES — nota N2 (0,75): só describe() + moda, sem nenhum gráfico

# DEPOIS — nota N3 (1,50): gráficos com matplotlib

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Vendas por categoria
vendas_por_categoria.plot(
    kind='bar',
    x='Categoria de Produto',
    y='Quantidade Total de Vendas',
    ax=axes[0],
    legend=False,
    color='steelblue'
)
axes[0].set_title('Volume de Vendas por Categoria')
axes[0].set_xlabel('Categoria')
axes[0].set_ylabel('Quantidade de Vendas')
axes[0].tick_params(axis='x', rotation=30)

# Gráfico 2: Vendas por mês
vendas_por_mes.plot(
    kind='line',
    x='Mês',
    y='Quantidade de Vendas',
    ax=axes[1],
    marker='o',
    color='coral'
)
axes[1].set_title('Evolução Mensal de Vendas')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('Quantidade de Vendas')

plt.tight_layout()
plt.show()
print("Gráficos gerados com sucesso!")

## Conclusões (igual ao original)

- A base de varejo analisada apresentou grande volume de registros e permitiu observar padrões importantes de consumo por gênero, categoria de produto e produto.
- A categoria `ALIMENTOS` foi a mais frequente na base, com destaque expressivo em relação às demais categorias.
- O público feminino apresentou maior quantidade total de registros de compra do que o público masculino.
- A análise da coluna `CL_FHL` mostrou concentração de clientes com poucos ou nenhum filho.
- Uma limitação importante da base é a ausência de uma coluna com valor monetário das vendas.

In [ ]:
print("=" * 60)
print("RESUMO — 4 AJUSTES PARA NOTA 10,00")
print("=" * 60)
print("  Versionamento         N2  →  1,25  (sem alteração)")
print("  Documentação          N2  →  1,25  (sem alteração)")
print("  CSV              N2 → N3  →  1,50  (+0,75) encoding + to_csv")
print("  Nulos            N2 → N3  →  1,50  (+0,75) fillna(mediana) para CL_FHL")
print("  Datas            N2 → N3  →  1,50  (+0,75) MES/ANO + groupby temporal")
print("  Agrupamentos          N3  →  1,50  (sem alteração)")
print("  Estatísticas     N2 → N3  →  1,50  (+0,75) gráficos matplotlib")
print("  " + "-" * 50)
print("  TOTAL:  7,00 → 10,00  (+3,00 pontos)")
print("=" * 60)

ERA SÓ ISSO.
SUCESSO MEU AMIGO, É SÓ CORRIGIR.